# Drug Utilisation with OMOPy

This notebook demonstrates `omopy.drug` for drug utilisation analysis on OMOP CDM data.

**Covered:**
- Drug cohort generation from concept sets
- Cohort requirements (first entry, washout, observation, date range)
- Drug use metrics (exposures, eras, days, quantities, doses)
- Daily dose calculation and pattern tables
- Indication and treatment intersection
- Summarise, table, and plot functions
- Drug restart analysis
- Dose coverage
- Mock data for richer demonstrations

**Database:** Synthea DuckDB with 27 persons, 663 drug_exposure records, 32 unique drug concepts.

**Note:** The Synthea database has an empty `drug_strength` table and NULL `quantity`/`sig` fields,
so daily dose calculations will produce NULLs. We use mock data to demonstrate those features.

## 1. Configuration

In [1]:
DB_PATH = "../data/synthea.duckdb"
CDM_SCHEMA = "base"
WRITE_SCHEMA = "main"

## 2. Setup

Connect to the CDM and import the drug utilisation module.

In [2]:
from omopy.connector import cdm_from_con, generate_concept_cohort_set
from omopy.generics import Codelist
from omopy.drug import (
    # Cohort generation
    generate_drug_utilisation_cohort_set,
    generate_ingredient_cohort_set,
    generate_atc_cohort_set,
    # Requirements
    require_is_first_drug_entry,
    require_prior_drug_washout,
    require_observation_before_drug,
    require_drug_in_date_range,
    # Drug use metrics
    add_drug_utilisation,
    add_number_exposures,
    add_number_eras,
    add_days_exposed,
    add_days_prescribed,
    add_time_to_exposure,
    add_initial_exposure_duration,
    add_initial_quantity,
    add_cumulative_quantity,
    add_initial_daily_dose,
    add_cumulative_dose,
    add_drug_restart,
    # Daily dose
    add_daily_dose,
    pattern_table,
    # Indication / treatment
    add_indication,
    add_treatment,
    # Summarise
    summarise_drug_utilisation,
    summarise_drug_restart,
    summarise_dose_coverage,
    summarise_indication,
    summarise_treatment,
    summarise_proportion_of_patients_covered,
    # Tables
    table_drug_utilisation,
    table_drug_restart,
    table_dose_coverage,
    table_indication,
    table_treatment,
    table_proportion_of_patients_covered,
    # Plots
    plot_drug_utilisation,
    plot_drug_restart,
    plot_indication,
    plot_treatment,
    plot_proportion_of_patients_covered,
    # Mock
    mock_drug_utilisation,
)

print("Imports OK")

Imports OK


In [3]:
cdm = cdm_from_con(DB_PATH, cdm_schema=CDM_SCHEMA, write_schema=WRITE_SCHEMA)
print(cdm)
print(f"Persons: {cdm['person'].count()}")

CdmReference(name='dbt-synthea', version=5.4, source=duckdb, tables=36)
Persons: 27


In [4]:
# Quick look at drug_exposure
de = cdm["drug_exposure"].collect()
print(f"Drug exposure records: {len(de)}")
print(f"Unique drug_concept_ids: {de['drug_concept_id'].n_unique()}")
print(f"Quantity non-null: {de['quantity'].drop_nulls().len()}")
de.head(5)

Drug exposure records: 663
Unique drug_concept_ids: 32
Quantity non-null: 0


drug_exposure_id,person_id,drug_concept_id,drug_exposure_start_date,drug_exposure_start_datetime,drug_exposure_end_date,drug_exposure_end_datetime,verbatim_end_date,drug_type_concept_id,stop_reason,refills,quantity,days_supply,sig,route_concept_id,lot_number,provider_id,visit_occurrence_id,visit_detail_id,drug_source_value,drug_source_concept_id,route_source_value,dose_unit_source_value
i64,i64,i32,date,datetime[μs],date,datetime[μs],date,i32,str,i32,i32,i64,str,i32,str,i64,i64,i64,str,i32,str,str
1,1,19125062,2014-08-02,2014-08-02 14:15:57,2014-08-02,2014-08-02 14:15:57,null,32838,null,null,null,null,null,0,"""0""",5,2,1000002,"""665078""",19125062,null,null
2,1,19019979,2023-04-22,2023-04-22 17:20:30,2023-05-10,2023-05-10 17:20:30,2023-05-10,32838,null,null,null,18,null,0,"""0""",5,4,1000004,"""198405""",19019979,null,null
3,1,1594382,2014-08-02,2014-08-02 14:15:57,2014-08-02,2014-08-02 14:15:57,null,32838,null,null,null,null,null,0,"""0""",5,2,1000002,"""1870230""",1594382,null,null
4,2,40160973,2020-10-07,2020-10-07 06:50:57,2020-10-07,2020-10-07 06:50:57,null,32838,null,null,null,null,null,0,"""0""",21,5,1000005,"""854235""",40160973,null,null
5,2,40160997,2020-10-14,2020-10-14 06:50:57,2020-10-14,2020-10-14 06:50:57,null,32838,null,null,null,null,null,0,"""0""",21,5,1000005,"""854252""",40160997,null,null


## 3. Generating Drug Cohorts

`generate_drug_utilisation_cohort_set` creates drug era cohorts from concept sets.
Records in `drug_exposure` matching the concept IDs are collapsed into eras
using a configurable gap (`gap_era` days).

In [5]:
# Define drug concept sets (common Synthea drugs)
# 1367500 = Amoxicillin, 19133768 = Ibuprofen
drug_concepts = Codelist({
    "amoxicillin": [1367500],
    "ibuprofen": [19133768],
})

cdm = generate_drug_utilisation_cohort_set(
    cdm,
    concept_set=drug_concepts,
    name="drug_cohort",
    gap_era=30,  # merge exposures within 30 days
)
print("Drug cohort generated:")
print(cdm["drug_cohort"])

Drug cohort generated:
CohortTable('drug_cohort', source='duckdb', cohorts=2)


In [6]:
# Inspect cohort settings and counts
print("Settings:")
print(cdm["drug_cohort"].settings)
print("\nCounts:")
print(cdm["drug_cohort"].cohort_count())

Settings:
shape: (2, 3)
┌──────────────────────┬─────────────┬─────────┐
│ cohort_definition_id ┆ cohort_name ┆ gap_era │
│ ---                  ┆ ---         ┆ ---     │
│ i64                  ┆ str         ┆ i64     │
╞══════════════════════╪═════════════╪═════════╡
│ 1                    ┆ amoxicillin ┆ 30      │
│ 2                    ┆ ibuprofen   ┆ 30      │
└──────────────────────┴─────────────┴─────────┘

Counts:
shape: (1, 3)
┌──────────────────────┬────────────────┬─────────────────┐
│ cohort_definition_id ┆ number_records ┆ number_subjects │
│ ---                  ┆ ---            ┆ ---             │
│ i64                  ┆ u32            ┆ u32             │
╞══════════════════════╪════════════════╪═════════════════╡
│ 2                    ┆ 1              ┆ 1               │
└──────────────────────┴────────────────┴─────────────────┘


In [7]:
# Attrition shows how many records survived each step
print("Attrition:")
print(cdm["drug_cohort"].attrition)

Attrition:
shape: (2, 7)
┌──────────────┬──────────────┬──────────────┬───────────┬─────────────┬─────────────┬─────────────┐
│ cohort_defin ┆ number_recor ┆ number_subje ┆ reason_id ┆ reason      ┆ excluded_re ┆ excluded_su │
│ ition_id     ┆ ds           ┆ cts          ┆ ---       ┆ ---         ┆ cords       ┆ bjects      │
│ ---          ┆ ---          ┆ ---          ┆ i64       ┆ str         ┆ ---         ┆ ---         │
│ i64          ┆ i64          ┆ i64          ┆           ┆             ┆ i64         ┆ i64         │
╞══════════════╪══════════════╪══════════════╪═══════════╪═════════════╪═════════════╪═════════════╡
│ 1            ┆ 0            ┆ 0            ┆ 1         ┆ Initial     ┆ 0           ┆ 0           │
│              ┆              ┆              ┆           ┆ qualifying  ┆             ┆             │
│              ┆              ┆              ┆           ┆ events      ┆             ┆             │
│ 2            ┆ 1            ┆ 1            ┆ 1         ┆ Initial

## 4. Applying Requirements

Requirements are filters applied to the drug cohort.
Each records an attrition row so you can track how many subjects/records are excluded.

In [8]:
# Keep only the first drug entry per person per cohort
cohort_first = require_is_first_drug_entry(cdm["drug_cohort"])
print("After require_is_first_drug_entry:")
print(cohort_first)
print(cohort_first.cohort_count())

After require_is_first_drug_entry:
CohortTable('drug_cohort', source='duckdb', cohorts=2)
shape: (1, 3)
┌──────────────────────┬────────────────┬─────────────────┐
│ cohort_definition_id ┆ number_records ┆ number_subjects │
│ ---                  ┆ ---            ┆ ---             │
│ i64                  ┆ u32            ┆ u32             │
╞══════════════════════╪════════════════╪═════════════════╡
│ 2                    ┆ 1              ┆ 1               │
└──────────────────────┴────────────────┴─────────────────┘


In [9]:
# Require a 365-day washout period (no prior drug entry within 365 days)
cohort_washout = require_prior_drug_washout(cdm["drug_cohort"], days=365)
print("After 365-day washout:")
print(cohort_washout)
print(cohort_washout.cohort_count())

After 365-day washout:
CohortTable('drug_cohort', source='duckdb', cohorts=2)
shape: (1, 3)
┌──────────────────────┬────────────────┬─────────────────┐
│ cohort_definition_id ┆ number_records ┆ number_subjects │
│ ---                  ┆ ---            ┆ ---             │
│ i64                  ┆ u32            ┆ u32             │
╞══════════════════════╪════════════════╪═════════════════╡
│ 2                    ┆ 1              ┆ 1               │
└──────────────────────┴────────────────┴─────────────────┘


In [10]:
# Require 180 days of prior observation before cohort start
cohort_obs = require_observation_before_drug(cdm["drug_cohort"], days=180)
print("After requiring 180 days prior observation:")
print(cohort_obs)
print(cohort_obs.cohort_count())

After requiring 180 days prior observation:
CohortTable('drug_cohort', source='duckdb', cohorts=2)
shape: (1, 3)
┌──────────────────────┬────────────────┬─────────────────┐
│ cohort_definition_id ┆ number_records ┆ number_subjects │
│ ---                  ┆ ---            ┆ ---             │
│ i64                  ┆ u32            ┆ u32             │
╞══════════════════════╪════════════════╪═════════════════╡
│ 2                    ┆ 1              ┆ 1               │
└──────────────────────┴────────────────┴─────────────────┘


In [11]:
# Restrict to entries within a date range
cohort_dated = require_drug_in_date_range(
    cdm["drug_cohort"],
    date_range=("2010-01-01", "2020-12-31"),
)
print("After date range restriction:")
print(cohort_dated)
print(cohort_dated.cohort_count())

After date range restriction:
CohortTable('drug_cohort', source='duckdb', cohorts=2)
shape: (0, 3)
┌──────────────────────┬────────────────┬─────────────────┐
│ cohort_definition_id ┆ number_records ┆ number_subjects │
│ ---                  ┆ ---            ┆ ---             │
│ i64                  ┆ u32            ┆ u32             │
╞══════════════════════╪════════════════╪═════════════════╡
└──────────────────────┴────────────────┴─────────────────┘


In [12]:
# Combined attrition after all requirements
print("Attrition for first-entry cohort:")
print(cohort_first.attrition)

Attrition for first-entry cohort:
shape: (3, 7)
┌──────────────┬──────────────┬──────────────┬───────────┬─────────────┬─────────────┬─────────────┐
│ cohort_defin ┆ number_recor ┆ number_subje ┆ reason_id ┆ reason      ┆ excluded_re ┆ excluded_su │
│ ition_id     ┆ ds           ┆ cts          ┆ ---       ┆ ---         ┆ cords       ┆ bjects      │
│ ---          ┆ ---          ┆ ---          ┆ i64       ┆ str         ┆ ---         ┆ ---         │
│ i64          ┆ i64          ┆ i64          ┆           ┆             ┆ i64         ┆ i64         │
╞══════════════╪══════════════╪══════════════╪═══════════╪═════════════╪═════════════╪═════════════╡
│ 1            ┆ 0            ┆ 0            ┆ 1         ┆ Initial     ┆ 0           ┆ 0           │
│              ┆              ┆              ┆           ┆ qualifying  ┆             ┆             │
│              ┆              ┆              ┆           ┆ events      ┆             ┆             │
│ 2            ┆ 1            ┆ 1          

## 5. Adding Drug Use Metrics

`add_drug_utilisation` adds all metrics at once. Individual functions
(e.g. `add_number_exposures`, `add_days_exposed`) can also be called separately.

**Note:** In this Synthea DB, `quantity` is always NULL and `drug_strength` is empty,
so quantity and dose columns will be NULL. The structural metrics
(number of exposures, eras, days) still work.

In [13]:
# Add all drug use metrics at once
enriched = add_drug_utilisation(
    cdm["drug_cohort"],
    gap_era=30,
    number_exposures=True,
    number_eras=True,
    days_exposed=True,
    days_prescribed=True,
    time_to_exposure=True,
    initial_exposure_duration=True,
    initial_quantity=True,
    cumulative_quantity=True,
    initial_daily_dose=False,  # skip dose — drug_strength is empty
    cumulative_dose=False,
)
print("Enriched cohort columns:")
print(enriched.columns)
enriched.collect().head(5)

Enriched cohort columns:
['cohort_definition_id', 'subject_id', 'cohort_start_date', 'cohort_end_date', 'number_exposures_ibuprofen', 'number_eras_ibuprofen', 'days_exposed_ibuprofen', 'days_prescribed_ibuprofen', 'time_to_exposure_ibuprofen', 'initial_exposure_duration_ibuprofen', 'initial_quantity_ibuprofen', 'cumulative_quantity_ibuprofen']


cohort_definition_id,subject_id,cohort_start_date,cohort_end_date,number_exposures_ibuprofen,number_eras_ibuprofen,days_exposed_ibuprofen,days_prescribed_ibuprofen,time_to_exposure_ibuprofen,initial_exposure_duration_ibuprofen,initial_quantity_ibuprofen,cumulative_quantity_ibuprofen
i64,i64,date,date,i64,i64,i64,i64,i64,i64,f64,f64
2,25,2004-03-21,2004-03-21,1,1,1,1,0,1,0.0,0.0


In [14]:
# Individual metric: number of exposures only
with_n_exp = add_number_exposures(cdm["drug_cohort"])
print("Number of exposures:")
with_n_exp.collect().head(5)

Number of exposures:


cohort_definition_id,subject_id,cohort_start_date,cohort_end_date,number_exposures_ibuprofen
i64,i64,date,date,i64
2,25,2004-03-21,2004-03-21,1


In [15]:
# Individual metric: number of eras
with_n_eras = add_number_eras(cdm["drug_cohort"], gap_era=30)
print("Number of eras:")
with_n_eras.collect().head(5)

Number of eras:


cohort_definition_id,subject_id,cohort_start_date,cohort_end_date,number_eras_ibuprofen
i64,i64,date,date,i64
2,25,2004-03-21,2004-03-21,1


In [16]:
# Individual metric: days exposed
with_days = add_days_exposed(cdm["drug_cohort"], gap_era=30)
print("Days exposed:")
with_days.collect().head(5)

Days exposed:


cohort_definition_id,subject_id,cohort_start_date,cohort_end_date,days_exposed_ibuprofen
i64,i64,date,date,i64
2,25,2004-03-21,2004-03-21,1


In [17]:
# Individual metric: initial exposure duration
with_dur = add_initial_exposure_duration(cdm["drug_cohort"])
print("Initial exposure duration:")
with_dur.collect().head(5)

Initial exposure duration:


cohort_definition_id,subject_id,cohort_start_date,cohort_end_date,initial_exposure_duration_ibuprofen
i64,i64,date,date,i64
2,25,2004-03-21,2004-03-21,1


## 6. Daily Dose & Patterns

`add_daily_dose` computes daily dose from `drug_strength`.
`pattern_table` inspects which drug strength patterns are present.

**Synthea limitation:** The `drug_strength` table is empty, so these will return
empty or all-NULL results. We demonstrate the API calls anyway.

In [18]:
# Inspect drug_strength patterns (empty in Synthea)
patterns = pattern_table(cdm)
print(f"Pattern table rows: {len(patterns)}")
patterns

Pattern table rows: 0


pattern_id,formula_name,unit,n
i32,str,str,i64


In [19]:
# add_daily_dose on drug_exposure — all NULLs expected
try:
    de_with_dose = add_daily_dose(
        cdm["drug_exposure"],
        cdm,
        ingredient_concept_id=1367500,  # amoxicillin
    )
    result = de_with_dose.collect()
    print(f"Rows: {len(result)}, daily_dose non-null: {result['daily_dose'].drop_nulls().len()}")
    result.select("drug_concept_id", "quantity", "daily_dose", "unit").head(5)
except Exception as e:
    print(f"Daily dose failed (expected with empty drug_strength): {e}")

Daily dose failed (expected with empty drug_strength): Catalog Error: Table with name __omopy_dose_patterns does not exist!
Did you mean "pg_prepared_statements"?


## 7. Indication & Treatment

`add_indication` checks whether subjects had a condition (indication) around
cohort start. `add_treatment` checks whether they were receiving another drug.

Both require a reference cohort (condition or drug) already registered in the CDM.

In [20]:
# Generate condition cohorts to use as indications
# 40481087 = Sinusitis (common in Synthea)
indication_concepts = Codelist({"sinusitis": [40481087]})
cdm = generate_concept_cohort_set(cdm, indication_concepts, name="indications")
print("Indication cohort:")
print(cdm["indications"])

Indication cohort:
CohortTable('indications', source='duckdb', cohorts=1)


In [21]:
# Add indication — was the patient diagnosed with sinusitis within
# 30 days before or on cohort start?
with_indication = add_indication(
    cdm["drug_cohort"],
    "indications",
    cdm=cdm,
    indication_window=(-30, 0),
    mutually_exclusive=True,
)
print("Cohort with indication:")
with_indication.collect().head(5)

Cohort with indication:


cohort_definition_id,subject_id,cohort_start_date,cohort_end_date,sinusitis_m30_to_0,indication_m30_to_0
i64,i64,date,date,i8,str
2,25,2004-03-21,2004-03-21,0,"""none"""


In [22]:
# Generate a second drug cohort for treatment analysis
# Check if patients on amoxicillin also received ibuprofen
treatment_concepts = Codelist({"ibuprofen": [19133768]})
cdm = generate_drug_utilisation_cohort_set(
    cdm,
    concept_set=treatment_concepts,
    name="treatment_cohort",
    gap_era=30,
)

# Add treatment — was the patient also receiving ibuprofen at cohort start?
with_treatment = add_treatment(
    cdm["drug_cohort"],
    "treatment_cohort",
    cdm=cdm,
    window=(-30, 30),
    mutually_exclusive=True,
)
print("Cohort with treatment:")
with_treatment.collect().head(5)

Cohort with treatment:


cohort_definition_id,subject_id,cohort_start_date,cohort_end_date,ibuprofen_m30_to_30,treatment_m30_to_30
i64,i64,date,date,i8,str
2,25,2004-03-21,2004-03-21,1,"""untreated"""


## 8. Summarise Drug Utilisation

`summarise_drug_utilisation` computes distribution statistics for all drug
use metrics. The result is a `SummarisedResult` that can be rendered as a
table or plot.

In [23]:
# Summarise drug utilisation metrics
sr_util = summarise_drug_utilisation(
    cdm["drug_cohort"],
    gap_era=30,
    number_exposures=True,
    number_eras=True,
    days_exposed=True,
    days_prescribed=True,
    initial_exposure_duration=True,
    initial_quantity=True,
    cumulative_quantity=True,
    initial_daily_dose=False,  # skip dose — empty drug_strength
    cumulative_dose=False,
)
print("Summarised drug utilisation:")
print(sr_util)

Summarised drug utilisation:
SummarisedResult(148 rows, 1 result_id(s))


In [24]:
# Render as a table
table_drug_utilisation(sr_util)

GT(_tbl_data=                variable_name variable_level  estimate_name  \
0              Number records                             N   
1             Number subjects                             N   
2              Number records                             N   
3             Number subjects                             N   
4            number exposures                     Mean (SD)   
..                        ...            ...            ...   
63            days prescribed                 Missing N (%)   
64           time to exposure                 Missing N (%)   
65  initial exposure duration                 Missing N (%)   
66           initial quantity                 Missing N (%)   
67        cumulative quantity                 Missing N (%)   

   [header_name]cdm_name\n[header_level]unknown  cohort_name concept_set  
0                                             0  amoxicillin           –  
1                                             0  amoxicillin           –  
2                                             1    ibuprofen           –  
3                                             1    ibuprofen           –  
4                                       NA (NA)  amoxicillin   ibuprofen  
..                                          ...          ...         ...  
63                                     0 (0.0%)    ibuprofen   ibuprofen  
64                                     0 (0.0%)    ibuprofen   ibuprofen  
65                                     0 (0.0%)    ibuprofen   ibuprofen  
66                                     0 (0.0%)    ibuprofen   ibuprofen  
67                                     0 (0.0%)    ibuprofen   ibuprofen  

[68 rows x 6 columns], _body=<great_tables._gt_data.Body object at 0x7c3418024c20>, _boxhead=Boxhead([ColInfo(var='variable_name', type=<ColInfoTypeEnum.default: 1>, column_label='variable_name', column_align='left', column_width=None), ColInfo(var='variable_level', type=<ColInfoTypeEnum.default: 1>, column_label='variable_level', column_align='right', column_width=None), ColInfo(var='estimate_name', type=<ColInfoTypeEnum.default: 1>, column_label='estimate_name', column_align='left', column_width=None), ColInfo(var='[header_name]cdm_name\n[header_level]unknown', type=<ColInfoTypeEnum.default: 1>, column_label='unknown', column_align='left', column_width=None), ColInfo(var='cohort_name', type=<ColInfoTypeEnum.default: 1>, column_label='cohort_name', column_align='left', column_width=None), ColInfo(var='concept_set', type=<ColInfoTypeEnum.default: 1>, column_label='concept_set', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7c3418024440>, _spanners=Spanners([SpannerInfo(spanner_id='cdm_name', spanner_level=0, spanner_label='cdm_name', spanner_units=None, spanner_pattern=None, vars=['[header_name]cdm_name\n[header_level]unknown'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7c3418025010>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7c3418176e90>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_level', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]cdm_name\n[header_level]unknown', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='cohort_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(col

In [25]:
# Plot drug utilisation
plot_drug_utilisation(sr_util)

## 9. Summarise Drug Restart

`summarise_drug_restart` classifies what happens after a drug era ends:
restart (same drug), switch (different drug), both, or untreated.
Requires a "switch" cohort representing the alternative drug(s).

In [26]:
# Summarise drug restart — does the patient restart amoxicillin
# or switch to ibuprofen within 180 days?
sr_restart = summarise_drug_restart(
    cdm["drug_cohort"],
    "treatment_cohort",
    cdm=cdm,
    follow_up_days=[180, 365],
)
print("Summarised drug restart:")
print(sr_restart)

Summarised drug restart:
SummarisedResult(8 rows, 1 result_id(s))


In [27]:
# Table and plot
table_drug_restart(sr_restart)

GT(_tbl_data=              variable_name variable_level estimate_name  \
0  Drug restart in 180 days      untreated         N (%)   
1  Drug restart in 365 days      untreated         N (%)   
2            Number records                        count   
3           Number subjects                        count   
4            Number records                        count   
5           Number subjects                        count   

  [header_name]cdm_name\n[header_level]unknown  cohort_name follow_up_days  
0                                   1 (100.0%)    ibuprofen            180  
1                                   1 (100.0%)    ibuprofen            365  
2                                            0  amoxicillin              –  
3                                            0  amoxicillin              –  
4                                            1    ibuprofen              –  
5                                            1    ibuprofen              –  , _body=<great_tables._gt_data.Body object at 0x7c331053fbb0>, _boxhead=Boxhead([ColInfo(var='variable_name', type=<ColInfoTypeEnum.row_group: 3>, column_label='variable_name', column_align='left', column_width=None), ColInfo(var='variable_level', type=<ColInfoTypeEnum.default: 1>, column_label='variable_level', column_align='left', column_width=None), ColInfo(var='estimate_name', type=<ColInfoTypeEnum.default: 1>, column_label='estimate_name', column_align='left', column_width=None), ColInfo(var='[header_name]cdm_name\n[header_level]unknown', type=<ColInfoTypeEnum.default: 1>, column_label='unknown', column_align='right', column_width=None), ColInfo(var='cohort_name', type=<ColInfoTypeEnum.default: 1>, column_label='cohort_name', column_align='left', column_width=None), ColInfo(var='follow_up_days', type=<ColInfoTypeEnum.default: 1>, column_label='follow_up_days', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7c32e02e0e10>, _spanners=Spanners([SpannerInfo(spanner_id='cdm_name', spanner_level=0, spanner_label='cdm_name', spanner_units=None, spanner_pattern=None, vars=['[header_name]cdm_name\n[header_level]unknown'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7c32e02e1450>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7c331053fce0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_level', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]cdm_name\n[header_level]unknown', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='cohort_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='follow_up_days', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_level', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]cdm_name\n[header_level]unknown', rownum=None, colnum=None, styles=[CellStyl

In [28]:
plot_drug_restart(sr_restart)

## 10. Summarise Indication & Treatment

Summarise the prevalence of indications and concomitant treatments across the drug cohort.

In [29]:
# Summarise indication
sr_indication = summarise_indication(
    cdm["drug_cohort"],
    "indications",
    cdm=cdm,
    indication_window=(-30, 0),
)
print("Summarised indication:")
print(sr_indication)

Summarised indication:
SummarisedResult(6 rows, 1 result_id(s))


In [30]:
table_indication(sr_indication)

GT(_tbl_data=         variable_name variable_level estimate_name  \
0  Indication m30_to_0           none         N (%)   
1       Number records                        count   
2      Number subjects                        count   
3       Number records                        count   
4      Number subjects                        count   

  [header_name]cdm_name\n[header_level]unknown  cohort_name window_name  
0                                   1 (100.0%)    ibuprofen    m30_to_0  
1                                            0  amoxicillin           –  
2                                            0  amoxicillin           –  
3                                            1    ibuprofen           –  
4                                            1    ibuprofen           –  , _body=<great_tables._gt_data.Body object at 0x7c262c2b97b0>, _boxhead=Boxhead([ColInfo(var='variable_name', type=<ColInfoTypeEnum.row_group: 3>, column_label='variable_name', column_align='left', column_width=None), ColInfo(var='variable_level', type=<ColInfoTypeEnum.default: 1>, column_label='variable_level', column_align='left', column_width=None), ColInfo(var='estimate_name', type=<ColInfoTypeEnum.default: 1>, column_label='estimate_name', column_align='left', column_width=None), ColInfo(var='[header_name]cdm_name\n[header_level]unknown', type=<ColInfoTypeEnum.default: 1>, column_label='unknown', column_align='right', column_width=None), ColInfo(var='cohort_name', type=<ColInfoTypeEnum.default: 1>, column_label='cohort_name', column_align='left', column_width=None), ColInfo(var='window_name', type=<ColInfoTypeEnum.default: 1>, column_label='window_name', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7c262c3e6fd0>, _spanners=Spanners([SpannerInfo(spanner_id='cdm_name', spanner_level=0, spanner_label='cdm_name', spanner_units=None, spanner_pattern=None, vars=['[header_name]cdm_name\n[header_level]unknown'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7c262c3e7100>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7c32c81ec950>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_level', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]cdm_name\n[header_level]unknown', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='cohort_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='window_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_level', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]cdm_name\n[header_level]unknown', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabe

In [31]:
plot_indication(sr_indication)

In [32]:
# Summarise treatment
sr_treatment = summarise_treatment(
    cdm["drug_cohort"],
    "treatment_cohort",
    cdm=cdm,
    window=(-30, 30),
)
print("Summarised treatment:")
print(sr_treatment)

Summarised treatment:
SummarisedResult(6 rows, 1 result_id(s))


In [33]:
table_treatment(sr_treatment)

GT(_tbl_data=          variable_name variable_level estimate_name  \
0  Medication m30_to_30      untreated         N (%)   
1        Number records                        count   
2       Number subjects                        count   
3        Number records                        count   
4       Number subjects                        count   

  [header_name]cdm_name\n[header_level]unknown  cohort_name window_name  
0                                   1 (100.0%)    ibuprofen   m30_to_30  
1                                            0  amoxicillin           –  
2                                            0  amoxicillin           –  
3                                            1    ibuprofen           –  
4                                            1    ibuprofen           –  , _body=<great_tables._gt_data.Body object at 0x7c331071c150>, _boxhead=Boxhead([ColInfo(var='variable_name', type=<ColInfoTypeEnum.row_group: 3>, column_label='variable_name', column_align='left', column_width=None), ColInfo(var='variable_level', type=<ColInfoTypeEnum.default: 1>, column_label='variable_level', column_align='left', column_width=None), ColInfo(var='estimate_name', type=<ColInfoTypeEnum.default: 1>, column_label='estimate_name', column_align='left', column_width=None), ColInfo(var='[header_name]cdm_name\n[header_level]unknown', type=<ColInfoTypeEnum.default: 1>, column_label='unknown', column_align='right', column_width=None), ColInfo(var='cohort_name', type=<ColInfoTypeEnum.default: 1>, column_label='cohort_name', column_align='left', column_width=None), ColInfo(var='window_name', type=<ColInfoTypeEnum.default: 1>, column_label='window_name', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7c262c3f31d0>, _spanners=Spanners([SpannerInfo(spanner_id='cdm_name', spanner_level=0, spanner_label='cdm_name', spanner_units=None, spanner_pattern=None, vars=['[header_name]cdm_name\n[header_level]unknown'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7c262c2b9bf0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7c262c2b99d0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_level', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]cdm_name\n[header_level]unknown', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='cohort_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='window_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_level', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]cdm_name\n[header_level]unknown', rownum=None, colnum=None, styles=[CellStyleText(color='#ffffff', font=None, size=None, align=None, v_align=None, style=None, weight='bold', stretch=None, decorate=None, transform=None, whitespace=None)]), StyleInfo(locname=LocColu

In [34]:
plot_treatment(sr_treatment)

## 11. Dose Coverage

`summarise_dose_coverage` reports the distribution of daily dose and the
missing-dose rate. Since `drug_strength` is empty in Synthea, all doses
will be missing (100%).

In [35]:
# Dose coverage — expect 100% missing in Synthea
try:
    sr_dose = summarise_dose_coverage(
        cdm["drug_cohort"],
        ingredient_concept_id=1367500,  # amoxicillin
    )
    print("Dose coverage:")
    print(sr_dose)
    table_dose_coverage(sr_dose)
except Exception as e:
    print(f"Dose coverage failed (expected with empty drug_strength): {e}")

Dose coverage failed (expected with empty drug_strength): 'Table' object has no attribute 'drug_concept_id'


## 12. Proportion of Patients Covered

`summarise_proportion_of_patients_covered` computes, for each day from 0
to `follow_up_days`, what fraction of patients are still covered by a drug era.

In [36]:
sr_ppc = summarise_proportion_of_patients_covered(
    cdm["drug_cohort"],
    follow_up_days=365,
)
print("Proportion of patients covered:")
print(sr_ppc)

Proportion of patients covered:
SummarisedResult(40 rows, 1 result_id(s))


In [37]:
table_proportion_of_patients_covered(sr_ppc)

GT(_tbl_data=   variable_name variable_level          estimate_name  \
0        overall                          PPC [95% CI]   
1        overall                          PPC [95% CI]   
2        overall                          PPC [95% CI]   
3        overall                          PPC [95% CI]   
4        overall                          PPC [95% CI]   
5        overall                          PPC [95% CI]   
6        overall                          PPC [95% CI]   
7        overall                          PPC [95% CI]   
8        overall                 N covered / N at risk   
9        overall                 N covered / N at risk   
10       overall                 N covered / N at risk   
11       overall                 N covered / N at risk   
12       overall                 N covered / N at risk   
13       overall                 N covered / N at risk   
14       overall                 N covered / N at risk   
15       overall                 N covered / N at risk   

   [header_name]cdm_name\n[header_level]dbt-synthea cohort_name time  
0                                1.00 [0.21 - 1.00]   ibuprofen    0  
1                                0.00 [0.00 - 0.79]   ibuprofen    1  
2                                0.00 [0.00 - 0.79]   ibuprofen    2  
3                                0.00 [0.00 - 0.79]   ibuprofen    3  
4                                0.00 [0.00 - 0.79]   ibuprofen    4  
5                                0.00 [0.00 - 0.79]   ibuprofen    5  
6                                0.00 [0.00 - 0.79]   ibuprofen    6  
7                                0.00 [0.00 - 0.79]   ibuprofen    7  
8                                             1 / 1   ibuprofen    0  
9                                             0 / 1   ibuprofen    1  
10                                            0 / 1   ibuprofen    2  
11                                            0 / 1   ibuprofen    3  
12                                            0 / 1   ibuprofen    4  
13                                            0 / 1   ibuprofen    5  
14                                            0 / 1   ibuprofen    6  
15                                            0 / 1   ibuprofen    7  , _body=<great_tables._gt_data.Body object at 0x7c262c366190>, _boxhead=Boxhead([ColInfo(var='variable_name', type=<ColInfoTypeEnum.default: 1>, column_label='variable_name', column_align='left', column_width=None), ColInfo(var='variable_level', type=<ColInfoTypeEnum.default: 1>, column_label='variable_level', column_align='right', column_width=None), ColInfo(var='estimate_name', type=<ColInfoTypeEnum.default: 1>, column_label='estimate_name', column_align='left', column_width=None), ColInfo(var='[header_name]cdm_name\n[header_level]dbt-synthea', type=<ColInfoTypeEnum.default: 1>, column_label='dbt-synthea', column_align='left', column_width=None), ColInfo(var='cohort_name', type=<ColInfoTypeEnum.default: 1>, column_label='cohort_name', column_align='left', column_width=None), ColInfo(var='time', type=<ColInfoTypeEnum.default: 1>, column_label='time', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7c25c02739b0>, _spanners=Spanners([SpannerInfo(spanner_id='cdm_name', spanner_level=0, spanner_label='cdm_name', spanner_units=None, spanner_pattern=None, vars=['[header_name]cdm_name\n[header_level]dbt-synthea'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7c3310713050>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7c3310725050>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_level', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=Loc

In [38]:
plot_proportion_of_patients_covered(sr_ppc)

## 13. Using Mock Data

The Synthea database is small (27 persons) with no `drug_strength` data.
`mock_drug_utilisation` generates a realistic `SummarisedResult` for
testing table and plot rendering without a database.

In [39]:
# Generate mock summarised result
mock_result = mock_drug_utilisation(n_cohorts=3, seed=42)
print("Mock result:")
print(mock_result)
mock_result.data.head(10)

Mock result:
SummarisedResult(276 rows, 1 result_id(s))


result_id,cdm_name,group_name,group_level,strata_name,strata_level,additional_name,additional_level,variable_name,variable_level,estimate_name,estimate_type,estimate_value
i64,str,str,str,str,str,str,str,str,str,str,str,str
1,"""mock_cdm""","""cohort_name""","""cohort_1""","""overall""","""overall""","""overall""","""overall""","""Number records""","""""","""count""","""integer""","""405"""
1,"""mock_cdm""","""cohort_name""","""cohort_1""","""overall""","""overall""","""overall""","""overall""","""Number subjects""","""""","""count""","""integer""","""377"""
1,"""mock_cdm""","""cohort_name""","""cohort_1""","""overall""","""overall""","""concept_set""","""drug_1""","""number exposures""","""""","""mean""","""numeric""","""1.23"""
1,"""mock_cdm""","""cohort_name""","""cohort_1""","""overall""","""overall""","""concept_set""","""drug_1""","""number exposures""","""""","""sd""","""numeric""","""0.26"""
1,"""mock_cdm""","""cohort_name""","""cohort_1""","""overall""","""overall""","""concept_set""","""drug_1""","""number exposures""","""""","""min""","""numeric""","""0.71"""
1,"""mock_cdm""","""cohort_name""","""cohort_1""","""overall""","""overall""","""concept_set""","""drug_1""","""number exposures""","""""","""q25""","""numeric""","""1.04"""
1,"""mock_cdm""","""cohort_name""","""cohort_1""","""overall""","""overall""","""concept_set""","""drug_1""","""number exposures""","""""","""median""","""numeric""","""1.23"""
1,"""mock_cdm""","""cohort_name""","""cohort_1""","""overall""","""overall""","""concept_set""","""drug_1""","""number exposures""","""""","""q75""","""numeric""","""1.41"""
1,"""mock_cdm""","""cohort_name""","""cohort_1""","""overall""","""overall""","""concept_set""","""drug_1""","""number exposures""","""""","""max""","""numeric""","""1.74"""


In [40]:
# Table from mock data — richer output than the small Synthea DB
table_drug_utilisation(mock_result)

GT(_tbl_data=                 variable_name variable_level  estimate_name  \
0               Number records                             N   
1              Number subjects                             N   
2               Number records                             N   
3              Number subjects                             N   
4               Number records                             N   
..                         ...            ...            ...   
121  initial exposure duration                 Missing N (%)   
122           initial quantity                 Missing N (%)   
123        cumulative quantity                 Missing N (%)   
124         initial daily dose                 Missing N (%)   
125            cumulative dose                 Missing N (%)   

    [header_name]cdm_name\n[header_level]mock_cdm cohort_name concept_set  
0                                             405    cohort_1           –  
1                                             377    cohort_1           –  
2                                             458    cohort_2           –  
3                                             366    cohort_2           –  
4                                             496    cohort_3           –  
..                                            ...         ...         ...  
121                                      2 (4.9%)    cohort_3      drug_1  
122                                      5 (0.6%)    cohort_3      drug_1  
123                                      5 (1.7%)    cohort_3      drug_1  
124                                      0 (4.8%)    cohort_3      drug_1  
125                                      4 (3.8%)    cohort_3      drug_1  

[126 rows x 6 columns], _body=<great_tables._gt_data.Body object at 0x7c25c03df110>, _boxhead=Boxhead([ColInfo(var='variable_name', type=<ColInfoTypeEnum.default: 1>, column_label='variable_name', column_align='left', column_width=None), ColInfo(var='variable_level', type=<ColInfoTypeEnum.default: 1>, column_label='variable_level', column_align='right', column_width=None), ColInfo(var='estimate_name', type=<ColInfoTypeEnum.default: 1>, column_label='estimate_name', column_align='left', column_width=None), ColInfo(var='[header_name]cdm_name\n[header_level]mock_cdm', type=<ColInfoTypeEnum.default: 1>, column_label='mock_cdm', column_align='left', column_width=None), ColInfo(var='cohort_name', type=<ColInfoTypeEnum.default: 1>, column_label='cohort_name', column_align='left', column_width=None), ColInfo(var='concept_set', type=<ColInfoTypeEnum.default: 1>, column_label='concept_set', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7c34d00d0f50>, _spanners=Spanners([SpannerInfo(spanner_id='cdm_name', spanner_level=0, spanner_label='cdm_name', spanner_units=None, spanner_pattern=None, vars=['[header_name]cdm_name\n[header_level]mock_cdm'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7c25c01234d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7c25c01e8140>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_level', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='estimate_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='[header_name]cdm_name\n[header_level]mock_cdm', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='cohort_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInf

In [41]:
# Plot from mock data
plot_drug_utilisation(mock_result)

In [42]:
# Mock with strata
mock_stratified = mock_drug_utilisation(n_cohorts=2, n_strata=2, seed=123)
print("Mock stratified result:")
print(mock_stratified)
table_drug_utilisation(mock_stratified)

Mock stratified result:
SummarisedResult(1104 rows, 1 result_id(s))


GT(_tbl_data=                 variable_name variable_level  estimate_name  \
0               Number records                             N   
1              Number subjects                             N   
2               Number records                             N   
3              Number subjects                             N   
4               Number records                             N   
..                         ...            ...            ...   
499  initial exposure duration                 Missing N (%)   
500           initial quantity                 Missing N (%)   
501        cumulative quantity                 Missing N (%)   
502         initial daily dose                 Missing N (%)   
503            cumulative dose                 Missing N (%)   

    [header_name]cdm_name\n[header_level]mock_cdm cohort_name age_group  \
0                                             144    cohort_1         –   
1                                              76    cohort_1         –   
2                                             608    cohort_1         –   
3                                             443    cohort_1         –   
4                                             265    cohort_1         –   
..                                            ...         ...       ...   
499                                      0 (3.1%)    cohort_2       >65   
500                                      3 (4.1%)    cohort_2       >65   
501                                      5 (0.9%)    cohort_2       >65   
502                                      3 (1.1%)    cohort_2       >65   
503                                      1 (4.0%)    cohort_2       >65   

        sex concept_set  
0         –           –  
1         –           –  
2      Male           –  
3      Male           –  
4    Female           –  
..      ...         ...  
499       –      drug_1  
500       –      drug_1  
501       –      drug_1  
502       –      drug_1  
503       –      drug_1  

[504 rows x 8 columns], _body=<great_tables._gt_data.Body object at 0x7c25c01abc20>, _boxhead=Boxhead([ColInfo(var='variable_name', type=<ColInfoTypeEnum.default: 1>, column_label='variable_name', column_align='left', column_width=None), ColInfo(var='variable_level', type=<ColInfoTypeEnum.default: 1>, column_label='variable_level', column_align='right', column_width=None), ColInfo(var='estimate_name', type=<ColInfoTypeEnum.default: 1>, column_label='estimate_name', column_align='left', column_width=None), ColInfo(var='[header_name]cdm_name\n[header_level]mock_cdm', type=<ColInfoTypeEnum.default: 1>, column_label='mock_cdm', column_align='left', column_width=None), ColInfo(var='cohort_name', type=<ColInfoTypeEnum.default: 1>, column_label='cohort_name', column_align='left', column_width=None), ColInfo(var='age_group', type=<ColInfoTypeEnum.default: 1>, column_label='age_group', column_align='left', column_width=None), ColInfo(var='sex', type=<ColInfoTypeEnum.default: 1>, column_label='sex', column_align='left', column_width=None), ColInfo(var='concept_set', type=<ColInfoTypeEnum.default: 1>, column_label='concept_set', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7c25c0138e50>, _spanners=Spanners([SpannerInfo(spanner_id='cdm_name', spanner_level=0, spanner_label='cdm_name', spanner_units=None, spanner_pattern=None, vars=['[header_name]cdm_name\n[header_level]mock_cdm'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7c25c0117cb0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7c25c0028210>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_name', rownum=None, colnum=None, styles=[CellStyleFill(color='#4361ee')]), StyleInfo(locname=LocColumnLabels(columns=None), grpname=None, colname='variable_level', rownum=None, colnum=None, style

In [43]:
plot_drug_utilisation(mock_stratified)

## 14. Summary

This notebook demonstrated the full `omopy.drug` workflow:

1. **Cohort generation** — `generate_drug_utilisation_cohort_set` creates drug era cohorts from concept sets
2. **Requirements** — `require_is_first_drug_entry`, `require_prior_drug_washout`, `require_observation_before_drug`, `require_drug_in_date_range` filter cohorts with attrition tracking
3. **Drug use metrics** — `add_drug_utilisation` (all-in-one) plus individual `add_*` functions for exposures, eras, days, quantities, and doses
4. **Daily dose** — `add_daily_dose` and `pattern_table` for dose calculation (requires populated `drug_strength`)
5. **Indication/treatment** — `add_indication` and `add_treatment` for clinical context
6. **Summarise** — `summarise_drug_utilisation`, `summarise_drug_restart`, `summarise_indication`, `summarise_treatment`, `summarise_dose_coverage`, `summarise_proportion_of_patients_covered`
7. **Tables & plots** — corresponding `table_*` and `plot_*` functions for each summary
8. **Mock data** — `mock_drug_utilisation` for testing without a database

**Key Synthea limitations:** empty `drug_strength` table and NULL `quantity`/`sig` fields mean
dose-related metrics are not available. Use mock data or a more complete CDM for those features.

**Next notebooks:**
- `08_large_scale_characteristics.ipynb` — large-scale characterisation across cohorts